In [1]:
import math
import sys
import yaml
sys.path.append('../../python/')  
from periphery import logicGate
from periphery import constant
from periphery.Technology import Technology
print(constant.INV)

0
0


In [8]:
with open('../../config.yaml', 'r') as file:
    config = yaml.safe_load(file)

with open('../../mapping.yaml', 'r') as file:
    mapping = yaml.safe_load(file)

with open('../../param.yaml', 'r') as file:
    param = yaml.safe_load(file)

In [9]:
class TSVPath:
    def __init__(self, tech, param, numRow, numCol):
        self.tech = tech
        self.param = param
        self.numRow = numRow
        self.numCol = numCol

        self.feature_size = tech.get_param('featureSize')
        self.temp = config['temperature']
        self.pnSizeRatio = self.tech.get_param('pnSizeRatio')
        self.vdd = tech.get_param('vdd')

        self.tsvPitch = self.param['tsvPitch']
        self.tsvRes = self.param['tsvRes']
        self.tsvCap = self.param['tsvCap']


        self.width_inv_n = constant.MIN_NMOS_SIZE * self.feature_size
        self.width_inv_p = self.pnSizeRatio * constant.MIN_NMOS_SIZE * self.feature_size
        self.initialized = True

    def calculate_area(self, new_height=None, new_width=None, option='NONE'):
        if not self.initialized:
            raise RuntimeError("[TSVPath] Error: Require initialization first!")

        h_inv, w_inv, _ = logicGate.calculate_logicgate_area(constant.INV, 1, self.width_inv_n, self.width_inv_p, constant.MAX_TRANSISTOR_HEIGHT * self.feature_size, self.tech)

        self.area = (2 * h_inv * w_inv + self.tsvPitch**2) * (self.numRow + self.numCol)

        self.cap_inv_input, self.cap_inv_output = logicGate.calculate_logicgate_cap(gate_type = constant.INV, num_Input = 1, width_NMOS = self.width_inv_n, width_PMOS = self.width_inv_p,
                                                                                     height_transistor_region = constant.MAX_TRANSISTOR_HEIGHT * self.feature_size, tech = self.tech)

        return self.area

    def calculate_latency(self, num_tsv, num_read):
        if not self.initialized:
            raise RuntimeError("[TSVPath] Error: Require initialization first!")

        ramp_input = 1e20
        #first inverter
        res_pull_up = logicGate.calculate_on_resistance(self.width_inv_p, constant.PMOS, self.temp, self.tech)
        tr1 = res_pull_up * (self.cap_inv_output + self.tsvCap * num_tsv)
        gm1 = logicGate.calculate_transconductance(self.width_inv_p, constant.PMOS, self.tech)
        beta1 = 1 / (res_pull_up * gm1)
        delay1, _ = logicGate.horowitz(tr1, beta1, ramp_input)

        #second inverter
        res_pull_down = self.tsvRes * num_tsv
        tr2 = res_pull_down * self.cap_inv_input
        delay2, _ = logicGate.horowitz(tr2, beta1, ramp_input)

        read_latency = (delay1 + delay2) * num_read

        # if self.param.get('synchronous', False):
        #     read_latency = math.ceil(read_latency * self.param['clkFreq'])

        return read_latency

    def calculate_power(self, num_tsv, num_read):
        if not self.initialized:
            raise RuntimeError("[TSVPath] Error: Require initialization first!")

        leakage = logicGate.calculate_logicgate_leakage(
            constant.INV, 1,
            self.width_inv_n,
            self.width_inv_p,
            self.temp,
            self.tech
        ) * self.vdd  * ( self.numRow + self.numCol)

        read_dynamic_energy = 0
        read_dynamic_energy += self.cap_inv_input * self.vdd**2
        read_dynamic_energy += (self.cap_inv_output + self.tsvCap) * self.vdd**2
        read_dynamic_energy += self.cap_inv_input * self.vdd**2
        read_dynamic_energy *= num_read

        if self.param.get('validated', False):
            read_dynamic_energy *= self.param['delta']

        return read_dynamic_energy, leakage

In [10]:
tech45 = Technology(node_nm=45, roadmap='HP')
# Instantiate and initialize Precharger
pre = TSVPath(
    param=param,
    tech=tech45,
    numRow=64,
    numCol=64
)

In [11]:
pre_charge_area = pre.calculate_area(
    new_height=None,
    new_width=None,
    option='NONE'
)
print("Area Result:", pre_charge_area)


Area Result: 4.8023552e-10


In [14]:
read_latency = pre.calculate_latency(
    num_tsv=10,
    num_read=1
)
print("Read Latency:", read_latency)

Read Latency: 2.294586085426847e-09


In [15]:
read_energy,leakage = pre.calculate_power(num_tsv=10, num_read=1)
print(f"  Read Dynamic Energy: {read_energy:.3e} J")
print(f"  Leakage Power: {leakage:.3e} W")

  Read Dynamic Energy: 2.059e-14 J
  Leakage Power: 2.787e-06 W
